In [1]:
import numpy as np
import open3d as o3d
from scipy.integrate import solve_ivp
from math import comb

# ---------- 1. 重新求解径向 ODE（稳定分支） ----------
n, k = 3, 2
R, r_max = 1.0, 10.0

def ode_system(r, y):
    u, v = y
    v_safe = max(v, 1e-5)
    dvdr = -((comb(n,k-1)/comb(n,k)) * (v_safe/r) * (1 - v_safe**(2*k)))
    return [v, dvdr]

sol = solve_ivp(ode_system, [R, r_max], [0.0, 0.99], 
                method='BDF', rtol=1e-8, atol=1e-8, max_step=0.05)

r_vals = sol.t
u_vals = sol.y[0]
v_vals = sol.y[1]

# ---------- 2. 生成 3D 旋转曲面点云 ----------
theta = np.linspace(0, 2*np.pi, 60)  # 60 个角度
points = []
for i in range(len(r_vals)):
    x = r_vals[i]
    y = u_vals[i]
    z = v_vals[i]
    # 绕 x 轴旋转：把 (x, y, z) 当作母线上的点，旋转生成圆环
    # 这里简化：把 (r, u(r), 0) 旋转，得到曲面上的点
    for th in theta:
        px = x
        py = y * np.cos(th)
        pz = y * np.sin(th)
        points.append([px, py, pz])

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(np.array(points))

# ---------- 3. 渲染 ----------
o3d.visualization.draw_geometries([pcd], window_name="Radial ODE Surface",
                                  width=800, height=600)
# ---------- 4. 法向估计（必须做，否则泊松重建会失败） ----------
pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
pcd.orient_normals_consistent_tangent_plane(k=15)  # 统一法向方向

# ---------- 5. 泊松重建 ----------
mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=9)
# 去除低密度顶点（噪声）
vertices_to_remove = densities < np.quantile(densities, 0.1)
mesh.remove_vertices_by_mask(vertices_to_remove)

o3d.visualization.draw_geometries([mesh], window_name="Poisson Reconstructed Mesh")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Open3D 核心数据结构（分支 A Day 1）

### PointCloud
- `o3d.geometry.PointCloud()`：空点云
- `pcd.points = o3d.utility.Vector3dVector(np_array)`：从 numpy 加载
- `pcd.estimate_normals()`：法向估计（KDTree 搜索）
- `pcd.orient_normals_consistent_tangent_plane(k)`：法向定向
- `o3d.io.write_point_cloud(filename, pcd)`：保存 .ply/.pcd

### TriangleMesh
- `o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth)`：泊松重建
- `mesh.remove_vertices_by_mask(mask)`：去除低密度顶点
- `mesh.compute_vertex_normals()`：计算顶点法向（用于着色）
- `o3d.io.write_triangle_mesh(filename, mesh)`：保存 .obj/.ply/.stl

### 可视化
- `o3d.visualization.draw_geometries([geometry_list])`：交互式窗口
- 快捷键：鼠标左键旋转，右键平移，滚轮缩放

### 与主线关联
- 图超曲面 $X(x)=(x,u(x))\subset\mathbb R^{n+1}$ 的离散化 = PointCloud → Poisson → Mesh
- 法向 $\nu$ 对应第二基本形式 $h_{ij}$ 的几何载体
- 曲率估计（主曲率 $\kappa$）可用 `o3d.geometry.TriangleMesh.compute_principal_curvatures()` 计算（待 Day 9 学）
- 

In [ ]:
o3d.io.write_triangle_mesh("radial_ode_mesh.obj", mesh)

True

In [ ]:
from IPython.display import Image, display
display(Image(filename="day8_radial_surface_pcd.png"))